# 台羅拼音轉語音檔下載（Taibun + Hapsing）

這份 notebook 示範如何輸入台羅拼音或台語漢字，透過 `taibun` 轉成數字調台羅，再呼叫 Hapsing 語音服務產生 MP3，最後在 notebook 中顯示播放器與下載連結。

這個版本預設不需要 Azure Speech 金鑰。

適合對象：
- 需要快速測試台羅 TTS 的開發者。
- 需要產生問卷題目或測驗提示音檔的專案。
- 想讓使用者輸入一句話後下載 MP3 的原型。

前置需求：
- 已安裝 `taibun` 與 `httpx`。
- 可以連線到 `https://hapsing.ithuan.tw/bangtsam`。

注意：Hapsing 是外部語音服務，正式產品使用前要確認穩定性、授權與流量限制。


## 流程

1. 載入專案內的 `scripts/tailo_tts_download.py`。
2. 使用 `taibun` 把輸入轉成 Hapsing 需要的數字調台羅。
3. 呼叫 Hapsing 產生 MP3。
4. 顯示音訊播放器與下載連結。
5. 封裝成函式，方便重複使用。


## 1. 載入共用函式

這個 cell 會找到專案根目錄、建立音檔輸出資料夾，並匯入剛才更新好的 script。


In [1]:
from __future__ import annotations

import importlib.util
from datetime import datetime
from pathlib import Path

from IPython.display import Audio, FileLink, display


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    return current


PROJECT_ROOT = find_project_root()
SCRIPT_PATH = PROJECT_ROOT / "scripts" / "tailo_tts_download.py"
AUDIO_DIR = PROJECT_ROOT / "output" / "jupyter-notebook" / "audio"
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

spec = importlib.util.spec_from_file_location("tailo_tts_download", SCRIPT_PATH)
tailo_tts = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(tailo_tts)


def display_audio_download(path: Path) -> None:
    resolved_path = path.resolve()
    display(Audio(filename=str(resolved_path)))
    link_path = resolved_path
    for base in (Path.cwd().resolve(), PROJECT_ROOT.resolve()):
        try:
            link_path = resolved_path.relative_to(base)
            break
        except ValueError:
            continue
    display(FileLink(str(link_path), result_html_prefix="下載音檔："))


print({"project_root": str(PROJECT_ROOT), "hapsing_endpoint": tailo_tts.HAPSING_ENDPOINT})


{'project_root': 'D:\\Project\\CogScreen-AI', 'hapsing_endpoint': 'https://hapsing.ithuan.tw/bangtsam'}


## 2. 輸入文字並轉成數字調台羅

你可以輸入台羅拼音，也可以輸入台語漢字。`taibun` 會先轉成 Hapsing 需要的數字調台羅，例如 `Lí hó.` 會變成 `li2 ho2.`。

`DIALECT` 可設定為 `south`、`north` 或 `singapore`。預設使用 `south`。


In [2]:
INPUT_TEXT = "Lí hó, guá beh tshì-khuànn Tâi-gí gi-im."
DIALECT = "south"
SANDHI = "none"

normalized_tailo = tailo_tts.normalize_tailo_for_hapsing(
    INPUT_TEXT,
    dialect=DIALECT,
    sandhi=SANDHI,
)

print("原始輸入：", INPUT_TEXT)
print("送給 Hapsing 的數字調台羅：", normalized_tailo)


原始輸入： Lí hó, guá beh tshì-khuànn Tâi-gí gi-im.
送給 Hapsing 的數字調台羅： li2 ho2, gua2 beh4 tshi3-khuann3 tai5-gi2 gi1-im1.


## 3. 產生 MP3 並提供下載

這個 cell 會呼叫 Hapsing，將音訊內容存成 MP3。若你的 Python 環境遇到 Hapsing TLS 憑證鏈問題，script 的 `tls_verify="auto"` 會先正常驗證，失敗時才對這個無金鑰音訊請求重試。


In [3]:
OUTPUT_PATH = AUDIO_DIR / "tailo-hapsing-demo.mp3"

saved_path = tailo_tts.synthesize_tailo_to_file(
    tailo_text=INPUT_TEXT,
    output_path=OUTPUT_PATH,
    provider="hapsing",
    dialect=DIALECT,
    sandhi=SANDHI,
    tls_verify="auto",
)

print(f"已產生音檔：{saved_path}")
print(f"檔案大小：{saved_path.stat().st_size:,} bytes")
display_audio_download(saved_path)


已產生音檔：D:\Project\CogScreen-AI\output\jupyter-notebook\audio\tailo-hapsing-demo.mp3
檔案大小：8,973 bytes


d:\Project\CogScreen-AI\output\jupyter-notebook\audio\tailo-hapsing-demo.mp3

## 4. 讓使用者輸入台羅拼音

執行下一個 cell 後，會出現輸入框。輸入台羅拼音或台語漢字並按 Enter，就會產生新的 MP3 與下載連結。


In [ ]:
USER_INPUT_TEXT = input("請輸入台羅拼音或台語漢字：").strip()
if not USER_INPUT_TEXT:
    raise ValueError("請輸入至少一個字串。")

timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
user_output_path = AUDIO_DIR / f"tailo-hapsing-user-{timestamp}.mp3"

print("送給 Hapsing 的數字調台羅：")
print(tailo_tts.normalize_tailo_for_hapsing(USER_INPUT_TEXT, dialect=DIALECT, sandhi=SANDHI))

saved_path = tailo_tts.synthesize_tailo_to_file(
    tailo_text=USER_INPUT_TEXT,
    output_path=user_output_path,
    provider="hapsing",
    dialect=DIALECT,
    sandhi=SANDHI,
)

print(f"已產生音檔：{saved_path.name}")
display_audio_download(saved_path)


## 5. 封裝成可重用函式

`make_tailo_download()` 可以在其他 cell 重複呼叫。它會回傳產生的 MP3 路徑，並顯示播放器與下載連結。


In [ ]:
def make_tailo_download(
    text: str,
    filename_prefix: str = "tailo-hapsing",
    dialect: str = DIALECT,
    sandhi: str = SANDHI,
) -> Path:
    text = text.strip()
    if not text:
        raise ValueError("輸入文字不可為空。")

    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    output_path = AUDIO_DIR / f"{filename_prefix}-{timestamp}.mp3"

    normalized = tailo_tts.normalize_tailo_for_hapsing(
        text,
        dialect=dialect,
        sandhi=sandhi,
    )
    print("送給 Hapsing 的數字調台羅：", normalized)

    saved_path = tailo_tts.synthesize_tailo_to_file(
        tailo_text=text,
        output_path=output_path,
        provider="hapsing",
        dialect=dialect,
        sandhi=sandhi,
    )

    print(f"已產生音檔：{saved_path}")
    display_audio_download(saved_path)
    return saved_path


## 練習

把下一個 cell 的內容改成你自己的句子，然後重新執行。建議分別測試：
- 含聲調符號的台羅。
- 台語漢字。
- 較長句子與標點符號。


In [ ]:
practice_text = "Guá beh lim tsuí. Lí beh tsia̍h pn̄g bô?"
practice_audio = make_tailo_download(practice_text, filename_prefix="tailo-practice")
practice_audio


## 命令列用法

如果不想開 notebook，也可以直接執行 script。預設 provider 是 `hapsing`，不需要 Azure 金鑰：

```bash
python scripts/tailo_tts_download.py "Lí hó." --output output/jupyter-notebook/audio/tailo-demo.mp3 --show-normalized
```

也可以輸入台語漢字：

```bash
python scripts/tailo_tts_download.py "你好" --output output/jupyter-notebook/audio/han-demo.mp3 --show-normalized
```

若要明確使用 Azure 備援，才需要指定 `--provider azure` 並設定 `SPEECH_KEY`、`SPEECH_REGION`。


## 常見問題

如果出現 `Missing dependency: taibun`，請執行：

```bash
pip install taibun
```

如果出現 Hapsing 連線錯誤，請先確認可以開啟 `https://hapsing.ithuan.tw/bangtsam?taibun=li2%20ho2`。這個端點是外部服務，可能會有暫時無法連線或回應變慢的情況。
